#Data Collection Pipeline for Hyderabad Air Quality Forecasting

- In this notebook, we build the dataset required for our air quality forecasting project.

- Instead of using a preprocessed dataset, we collect historical pollutant measurements directly from the OpenAQ API. This gives us greater control over the data quality, time range, and pollutants included.

###The pipeline consists of:

- Installing required libraries.
- Configuring the OpenAQ API.
- Discovering monitoring locations in Hyderabad.
- Retrieving pollutant sensors for each location.
- Downloading historical daily measurements.
- Transforming the raw data into a machine-learning-ready format.
- Cleaning missing values and invalid readings.
- Computing AQI using CPCB breakpoint methodology.

#Why Not Use the Government Downloaded Dataset?

- Initially, the official government dataset was explored.

- However, it had several limitations:

- It was primarily a snapshot export rather than a historical time series.
- Pollutants were stored in long format.
- AQI values were not directly available.
- Missing values existed for several pollutants.

- Therefore, OpenAQ was selected to programmatically collect historical measurements

#Step 1: Install Required Libraries

- This cell installs and imports the Python packages needed throughout the notebook.

- requests → Sends HTTP requests to the OpenAQ API.
- pandas → Stores and manipulates tabular data.
- tqdm → Displays progress bars while downloading large amounts of data.
- datetime → Handles dates used for historical queries.
- os → Creates directories and manages file paths.

In [49]:
# Install dependencies
!pip install requests pandas tqdm -q

import requests, pandas as pd, time, os
from tqdm import tqdm
from datetime import datetime, timedelta
import os
os.makedirs("data/processed", exist_ok=True)
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

#Step 2: Configure the API

###Before requesting data, we define:

- The API key used for authentication.
- The base URL of the OpenAQ service.
- HTTP headers required for authorization.
- The pollutants of interest.

###The project focuses on pollutants commonly used in air quality assessment:

- `PM2.5`
- `PM10`
- `NO₂`
- `SO₂`
- `CO`
- `O₃`
- `NH₃`

- Filtering to these pollutants avoids downloading unnecessary measurements and keeps the dataset relevant.

In [ ]:
# ── Paste your key here only ──────────────────────────────
API_KEY = "API_KEY_HERE"
# ─────────────────────────────────────────────────────────

BASE    = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}
CITY    = "Hyderabad"
COUNTRY = "IN"

# Pollutants we care about (CPCB AQI uses these 7)
TARGET_PARAMS = {"pm25", "pm10", "no2", "so2", "co", "o3", "nh3"}

os.makedirs("data/raw", exist_ok=True)
print("Setup done.")

Setup done.


#Step 3: Discover Monitoring Locations

- The OpenAQ API supports searching by geographic bounding boxes.

- A bounding box covering Hyderabad is supplied to retrieve all monitoring stations inside the city.

- The function loops through multiple API pages until every location has been collected.

###The output is saved as locations.csv, which contains:

- Location ID
- Station name
- First reporting date
- Number of sensors available

- This file acts as the foundation for all subsequent API requests.

In [51]:
# Hyderabad bounding box (covers city + outskirts)
# bbox = min_lon,min_lat,max_lon,max_lat
BBOX = "78.20,17.20,78.70,17.60"

def get_locations_bbox(bbox, limit=100):
    """Fetch locations within a bounding box."""
    locations, page = [], 1
    while True:
        while True:
            r = requests.get(
                f"{BASE}/locations",
                headers=HEADERS,
                params={"bbox": bbox, "limit": limit, "page": page}
            )
            if r.status_code == 429:
                print(f"Rate limited p{page}, waiting 10s...")
                time.sleep(10); continue
            r.raise_for_status()
            break

        results = r.json().get("results", [])
        if not results: break
        locations.extend(results)
        print(f"Page {page}: {len(locations)} locations", end="\r")
        if len(results) < limit: break
        page += 1
        time.sleep(0.5)
    return locations

locations = get_locations_bbox(BBOX)
print(f"\nFound {len(locations)} locations in Hyderabad bbox")

# Safe extraction — handles null datetimeFirst
def safe_date(l):
    dt = l.get("datetimeFirst")
    if not dt: return "?"
    local = dt.get("local")
    return local[:10] if local else "?"

loc_df = pd.DataFrame([{
    "id":      l["id"],
    "name":    l["name"],
    "since":   safe_date(l),
    "sensors": len(l.get("sensors", []))
} for l in locations])

print(loc_df.to_string(index=False))
loc_df.to_csv("data/raw/locations.csv", index=False)

Page 1: 18 locations
Found 18 locations in Hyderabad bbox
     id                                         name      since  sensors
    407                  Zoo Park, Hyderabad - TSPCB 2016-03-22       18
   2459                US Diplomatic Post: Hyderabad 2016-01-30        1
   2594               Sanathnagar, Hyderabad - TSPCB 2016-03-22        4
   5599  Bollaram Industrial Area, Hyderabad - TSPCB 2018-03-09       18
   5623        Central University, Hyderabad - TSPCB 2018-03-09       18
   5624          IDA Pashamylaram, Hyderabad - TSPCB 2018-03-09       18
   5645        ICRISAT Patancheru, Hyderabad - TSPCB 2018-03-11       18
   5647               Sanathnagar, Hyderabad - TSPCB 2018-03-09       16
   8557                                    Hyderabad 2016-11-10        1
  10897        ICRISAT Patancheru, Hyderabad - TSPCB 2020-04-28        6
 344103                ECIL Kapra, Hyderabad - TSPCB 2022-05-26       18
 344104 Kompally Municipal Office, Hyderabad - TSPCB 2022-05-26   

#Step 4: Retrieve Sensor Metadata

- Each monitoring location contains multiple sensors measuring different pollutants.

- This code requests sensor information for every location and filters the results to retain only pollutants relevant to this project.

- An additional filter keeps measurements reported in µg/m³, avoiding duplicate sensors reporting in alternative units such as ppb.

- The resulting metadata is saved as sensors.csv.

In [52]:
def get_sensors(location_id):
    """Get all sensors at a location, filter to AQI pollutants."""
    r = requests.get(
        f"{BASE}/locations/{location_id}/sensors",
        headers=HEADERS
    )
    if r.status_code != 200:
        return []
    return r.json().get("results", [])

all_sensors = []
for loc in tqdm(locations, desc="Fetching sensors"):
    sensors = get_sensors(loc["id"])
    for s in sensors:
        param = s.get("parameter", {}).get("name", "").lower()
        unit  = s.get("parameter", {}).get("units", "")
        # Keep only µg/m³ readings (not ppb duplicates)
        if param in TARGET_PARAMS and "µg" in unit:
            all_sensors.append({
                "sensor_id":   s["id"],
                "location_id": loc["id"],
                "location":    loc["name"],
                "pollutant":   param,
                "unit":        unit,
            })
    time.sleep(0.2)

sensor_df = pd.DataFrame(all_sensors)
print(f"\n{len(sensor_df)} usable sensors found")
print(sensor_df.groupby("pollutant")["sensor_id"].count())
sensor_df.to_csv("data/raw/sensors.csv", index=False)

Fetching sensors: 100%|██████████| 18/18 [00:07<00:00,  2.41it/s]


127 usable sensors found
pollutant
co      15
no2     14
o3      27
pm10    25
pm25    31
so2     15
Name: sensor_id, dtype: int64


#Step 5: Download Historical Measurements

- Each sensor has its own historical record.

- The function fetch_daily() requests daily aggregated measurements from 2016 until the present day.

- Important implementation details:

- Automatic pagination retrieves all available pages.
- API rate limits (HTTP 429) are handled by pausing and retrying.
- Measurements from every sensor are appended into a single dataset.

- The final output is stored as hyderabad_raw.csv.

In [53]:
def fetch_daily(sensor_id, date_from="2016-01-01", date_to=None):
    """Pull all daily averages for one sensor with pagination."""
    if date_to is None:
        date_to = datetime.today().strftime("%Y-%m-%d")
    records, page = [], 1
    while True:
        r = requests.get(
            f"{BASE}/sensors/{sensor_id}/days",
            headers=HEADERS,
            params={
                "date_from": date_from,
                "date_to":   date_to,
                "limit":     1000,
                "page":      page
            }
        )
        if r.status_code == 429:          # rate limited
            time.sleep(5); continue
        if r.status_code != 200:
            break
        results = r.json().get("results", [])
        if not results: break
        records.extend(results)
        if len(results) < 1000: break
        page += 1
        time.sleep(0.3)
    return records

all_records = []
for _, row in tqdm(sensor_df.iterrows(),
                    total=len(sensor_df),
                    desc="Downloading history"):
    records = fetch_daily(row["sensor_id"])
    for rec in records:
        all_records.append({
            "date":        rec["period"]["datetimeFrom"]["local"][:10],
            "location":    row["location"],
            "sensor_id":   row["sensor_id"],
            "pollutant":   row["pollutant"],
            "avg":         rec["value"],
            "unit":        row["unit"],
        })
    time.sleep(0.25)

raw_df = pd.DataFrame(all_records)
raw_df.to_csv("data/raw/hyderabad_raw.csv", index=False)
print(f"\nTotal records: {len(raw_df):,}")
print(f"Date range: {raw_df['date'].min()} → {raw_df['date'].max()}")
print(raw_df.groupby("pollutant")["avg"].describe().round(2))


Total records: 66,169
Date range: 2016-01-30 → 2026-06-11
             count     mean      std    min     25%     50%    75%       max
pollutant                                                                   
co          8195.0  1685.98  8777.11 -35.30  352.00  513.00  755.0  396000.0
no2         7549.0    29.15    22.48  -0.36   13.60   24.40   40.7     301.0
o3         13816.0    27.72    24.18 -38.80   15.60   23.60   33.3    1590.0
pm10       11685.0    81.95    40.13   0.00   55.30   79.00  103.0    1250.0
pm25       16963.0    40.20    25.96 -12.00   22.70   34.10   54.9     591.0
so2         7795.0   113.11  6978.25  -2.50    4.11    7.61   18.4  604000.0


#Step 6: Convert Long Format into Wide Format and Clean and Aggregate the Data

- The raw API returns one pollutant measurement per row.

###For example:

| Date       | Pollutant | Value |
|------------|-----------|------:|
| 2024-01-01 | PM2.5     |    45 |
| 2024-01-01 | PM10      |    82 |

- Machine learning models require all features in a single row.

- Therefore, the data is pivoted into:

| Date       | PM2.5 | PM10 | NO₂ | SO₂ | CO  | O₃  |
|------------|------:|-----:|----:|----:|----:|----:|
| 2024-01-01 | 45    | 82   | 18  | 6   | 0.7 | 32  |

- This transformation creates hyderabad_wide.csv, which is much easier to analyze and model.

###Several preprocessing operations are performed:

- Remove physically impossible values.
- Clip extreme outliers based on pollutant-specific limits.
- Average duplicate sensor readings.
- Aggregate multiple monitoring stations into a city-level average.
- Generate a continuous daily date index.
- Preserve missing observations explicitly.

- These steps improve consistency before imputation.

In [54]:


raw = pd.read_csv("data/raw/hyderabad_raw.csv", parse_dates=["date"])
print(f"Loaded {len(raw):,} records")

# ── Step 1: clip physically impossible values ─────────────
# Thresholds from CPCB instrument ranges + domain knowledge
LIMITS = {
    "pm25": (0, 500),
    "pm10": (0, 600),
    "no2":  (0, 400),
    "so2":  (0, 800),
    "co":   (0, 10000),
    "o3":   (0, 400),
}

before = len(raw)
for pol, (lo, hi) in LIMITS.items():
    mask = (raw["pollutant"] == pol)
    raw.loc[mask, "avg"] = raw.loc[mask, "avg"].clip(lower=lo, upper=hi)

# Set negatives to NaN (clipping to 0 would bias the mean)
raw.loc[raw["avg"] <= 0, "avg"] = np.nan
after_clean = raw["avg"].notna().sum()
print(f"Records after clipping: {after_clean:,} ({before - after_clean} set to NaN)")

# ── Step 2: average across sensors per station per day ────
# Multiple sensors of same pollutant at same location → mean
daily = (raw
    .groupby(["date", "location", "pollutant"])["avg"]
    .mean()
    .reset_index()
)

# ── Step 3: average across stations → single city-level value
city = (daily
    .groupby(["date", "pollutant"])["avg"]
    .mean()
    .reset_index()
)

# ── Step 4: pivot to wide format (one row per day) ────────
wide = city.pivot(index="date", columns="pollutant", values="avg")
wide.columns.name = None
wide = wide.reset_index()

# ── Step 5: fill full date range so gaps are explicit ─────
full_idx = pd.date_range(wide["date"].min(), wide["date"].max(), freq="D")
wide = wide.set_index("date").reindex(full_idx).rename_axis("date").reset_index()

print(f"\nShape: {wide.shape}")
print(f"Date range: {wide['date'].min().date()} → {wide['date'].max().date()}")
print(f"\nMissing % per pollutant:")
print((wide.drop(columns="date").isna().mean() * 100).round(1).to_string())
wide.to_csv("data/raw/hyderabad_wide.csv", index=False)

Loaded 66,169 records
Records after clipping: 65,684 (485 set to NaN)

Shape: (3786, 7)
Date range: 2016-01-30 → 2026-06-11

Missing % per pollutant:
co      51.4
no2     53.9
o3      39.3
pm10    41.3
pm25     4.8
so2     53.5




# Issue Discovered: High Missing Values After City-Level Aggregation

Initially, pollutant measurements from all Hyderabad monitoring stations were aggregated into a single city-level time series. During exploratory analysis, this approach revealed unexpectedly high percentages of missing values for several pollutants.

| Pollutant | Approximate Missing % |
| --------- | --------------------: |
| PM2.5     |                   ~5% |
| PM10      |                  ~41% |
| O₃        |                  ~39% |
| CO        |                  ~51% |
| NO₂       |                  ~54% |
| SO₂       |                  ~54% |

This raised concerns about the reliability of subsequent imputation and AQI computation.

### Root Cause

- The missing values were **not caused by sensor failures alone**. Different monitoring stations measure different subsets of pollutants and have different operational periods.

- By averaging all stations into a single city-wide record, dates where one station lacked a pollutant created unnecessary missing values in the aggregated dataset.

### Solution

- Instead of treating Hyderabad as a single monitoring location, each station will be treated as an independent time series.

###Advantages of this approach:

* Preserves station-specific pollution patterns.
* Significantly reduces artificial missingness.
* Minimizes the need for aggressive imputation.
* Produces more realistic forecasting models.
* Enables station-level and city-level analyses.

#Step 7: Identify stations with acceptable data quality

- The next step is to identify stations with acceptable data quality and retain them for modeling.


In [55]:


# -----------------------------
# Load raw data
# -----------------------------
raw = pd.read_csv(
    "data/raw/hyderabad_raw.csv",
    parse_dates=["date"]
)

# -----------------------------
# Basic cleaning
# -----------------------------
raw["pollutant"] = raw["pollutant"].str.lower()

# Remove negative values
raw.loc[raw["avg"] < 0, "avg"] = np.nan

# -----------------------------
# Aggregate duplicate readings
# (same station + same day + same pollutant)
# -----------------------------
daily = (
    raw.groupby(
        ["date", "location", "pollutant"],
        as_index=False
    )["avg"]
    .mean()
)

# -----------------------------
# Pivot to wide format
# -----------------------------
station_df = (
    daily.pivot(
        index=["date", "location"],
        columns="pollutant",
        values="avg"
    )
    .reset_index()
)

# Remove MultiIndex name
station_df.columns.name = None

# -----------------------------
# Sort chronologically
# -----------------------------
station_df = station_df.sort_values(
    ["location", "date"]
).reset_index(drop=True)



# -----------------------------
# Save
# -----------------------------
station_df.to_csv(
    "data/processed/hyderabad_station_level.csv",
    index=False
)

print("Saved: data/processed/hyderabad_station_level.csv")


print(station_df.shape)
station_df.head()

Saved: data/processed/hyderabad_station_level.csv
(17542, 8)


,date,location,co,no2,o3,pm10,pm25,so2
0,2018-03-09,"Bollaram Industrial Area, Hyderabad - TSPCB",51500.0,40.0,38.0,118.0,106.0,10.0
1,2018-03-10,"Bollaram Industrial Area, Hyderabad - TSPCB",43200.0,44.8,33.8,121.0,108.0,10.0
2,2018-03-11,"Bollaram Industrial Area, Hyderabad - TSPCB",35200.0,45.2,52.4,135.0,146.0,12.2
3,2018-03-12,"Bollaram Industrial Area, Hyderabad - TSPCB",23000.0,25.3,44.4,97.1,81.9,8.0
4,2018-03-13,"Bollaram Industrial Area, Hyderabad - TSPCB",27700.0,34.0,26.3,92.0,66.3,7.0


In [56]:
# Calculate percentage of missing values for each station

missing_pct = (
    station_df
    .groupby("location")
    .apply(lambda x: x.drop(columns=["date", "location"]).isna().mean() * 100)
)

missing_pct.round(2)

/tmp/ipykernel_2693/3287923242.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.drop(columns=["date", "location"]).isna().mean() * 100)


,co,no2,o3,pm10,pm25,so2
location,,,,,,
"Bollaram Industrial Area, Hyderabad - TSPCB",26.67,25.40,1.27,0.49,0.35,25.55
"Central University, Hyderabad - TSPCB",28.51,28.57,0.39,1.35,0.58,28.57
"ECIL Kapra, Hyderabad - TSPCB",85.19,85.19,0.00,0.00,0.00,85.19
Hyderabad,100.00,100.00,100.00,100.00,5.16,100.00
"ICRISAT Patancheru, Hyderabad - TSPCB",28.96,29.02,0.06,0.70,0.51,28.96
"IDA Pashamylaram, Hyderabad - TSPCB",29.73,33.63,0.55,1.51,1.64,29.79
"Kokapet, Hyderabad - TSPCB",84.15,84.15,15.85,0.00,0.00,84.15
"Kompally Municipal Office, Hyderabad - TSPCB",83.33,83.33,0.00,0.00,0.00,83.33
"Nacharam_TSIIC IALA, Hyderabad - TSPCB",82.77,82.77,0.19,0.00,0.00,82.77


# Step 8 : Selecting High-Quality Monitoring Stations


- After converting the raw OpenAQ data into a station-level dataset, a station-wise data quality assessment was performed to evaluate the completeness of pollutant measurements.

- Although some stations had nearly complete observations for **PM2.5** and **PM10**, many of them exhibited extremely high missing values for gaseous pollutants such as **CO**, **NO₂**, and **SO₂**.

For example:

| Station | PM2.5 | PM10 | CO | NO₂ | SO₂ |
|----------|-------|-------|------|------|------|
| ECIL Kapra | 0% | 0% | 85.16% | 85.16% | 85.16% |
| Kokapet | 0% | 0% | 84.12% | 84.12% | 84.12% |
| Kompally | 0% | 0% | 83.30% | 83.30% | 83.30% |

Retaining such stations would require extensive imputation and could introduce artificial patterns into the dataset.

## Selection Criteria

To build a reliable multi-pollutant forecasting dataset, stations are retained only if they satisfy:

- PM2.5 missing percentage ≤ **5%**
- PM10 missing percentage ≤ **10%**
- CO missing percentage ≤ **35%**
- NO₂ missing percentage ≤ **35%**
- SO₂ missing percentage ≤ **35%**
- O₃ missing percentage ≤ **10%**

| Pollutant | Threshold | Reason |
|-----------|-----------|--------|
| PM2.5 | ≤ 5% | Highest health impact, primary AQI driver |
| O₃ | ≤ 10% | Nonlinear peaks, second most critical |
| PM10 | ≤ 10% | Important but secondary to PM2.5 |
| CO/NO₂/SO₂ | ≤ 35% | Episodic, secondary AQI role, sensor drift common |

-These thresholds preserve stations with strong overall data quality while minimizing the need for aggressive imputation.

## Selected Stations

Based on the above criteria, the following monitoring stations are retained:

- Bollaram Industrial Area
- Central University
- ICRISAT Patancheru
- IDA Pashamylaram
- Zoo Park

These stations provide the best balance between pollutant coverage and data completeness and will be used for all subsequent analyses and model development.

In [57]:
# Calculate missing percentage for each station
station_quality = (
    station_df.groupby("location")
    .agg(
        pm25_missing=("pm25", lambda x: x.isna().mean() * 100),
        pm10_missing=("pm10", lambda x: x.isna().mean() * 100),
        co_missing=("co", lambda x: x.isna().mean() * 100),
        no2_missing=("no2", lambda x: x.isna().mean() * 100),
        so2_missing=("so2", lambda x: x.isna().mean() * 100),
        o3_missing=("o3", lambda x: x.isna().mean() * 100),
    )
    .round(2)
)

# Apply selection criteria
good_stations = station_quality[
    (station_quality["pm25_missing"] <= 5) &
    (station_quality["pm10_missing"] <= 10) &
    (station_quality["co_missing"] <= 35) &
    (station_quality["no2_missing"] <= 35) &
    (station_quality["so2_missing"] <= 35) &
    (station_quality["o3_missing"] <= 10)
]



# Filter the original dataset
station_filtered = station_df[
    station_df["location"].isin(good_stations.index)
].copy()

station_filtered = station_filtered.sort_values(
    ["location", "date"]
).reset_index(drop=True)

print(f"\nFiltered dataset shape: {station_filtered.shape}")

# Save the filtered dataset
station_filtered.to_csv(
    "data/processed/hyderabad_station_filtered.csv",
    index=False
)

print("\nSaved: data/processed/hyderabad_station_filtered.csv")

print("Selected Stations:")
good_stations


Filtered dataset shape: (8077, 8)

Saved: data/processed/hyderabad_station_filtered.csv
Selected Stations:


,pm25_missing,pm10_missing,co_missing,no2_missing,so2_missing,o3_missing
location,,,,,,
"Bollaram Industrial Area, Hyderabad - TSPCB",0.35,0.49,26.67,25.40,25.55,1.27
"Central University, Hyderabad - TSPCB",0.58,1.35,28.51,28.57,28.57,0.39
"ICRISAT Patancheru, Hyderabad - TSPCB",0.51,0.70,28.96,29.02,28.96,0.06
"IDA Pashamylaram, Hyderabad - TSPCB",1.64,1.51,29.73,33.63,29.79,0.55
"Zoo Park, Hyderabad - TSPCB",2.23,5.14,22.24,23.06,22.24,5.14


# Step 9 : Final Dataset Construction

- After restructuring the raw API responses into a station-level format, the resulting dataset contains one row per **monitoring station per day**.

- Unlike the earlier city-level aggregation approach, this design preserves spatial information and minimizes unnecessary missing values.

###The dataset includes:

* `date` – Observation date
* `location` – Monitoring station
* `pm25` – PM2.5 concentration
* `pm10` – PM10 concentration
* `co` – Carbon monoxide concentration
* `no2` – Nitrogen dioxide concentration
* `so2` – Sulfur dioxide concentration
* `o3` – Ozone concentration

This station-level dataset will serve as the foundation for exploratory analysis, feature engineering, forecasting, and explainability.


In [58]:
df=pd.read_csv("data/processed/hyderabad_station_filtered.csv")

In [59]:
df.head()

,date,location,co,no2,o3,pm10,pm25,so2
0,2018-03-09,"Bollaram Industrial Area, Hyderabad - TSPCB",51500.0,40.0,38.0,118.0,106.0,10.0
1,2018-03-10,"Bollaram Industrial Area, Hyderabad - TSPCB",43200.0,44.8,33.8,121.0,108.0,10.0
2,2018-03-11,"Bollaram Industrial Area, Hyderabad - TSPCB",35200.0,45.2,52.4,135.0,146.0,12.2
3,2018-03-12,"Bollaram Industrial Area, Hyderabad - TSPCB",23000.0,25.3,44.4,97.1,81.9,8.0
4,2018-03-13,"Bollaram Industrial Area, Hyderabad - TSPCB",27700.0,34.0,26.3,92.0,66.3,7.0


In [60]:
# Percentage of non-missing values
completeness = (
    df
    .groupby("location")[["pm25", "pm10", "co", "no2", "so2", "o3"]]
    .apply(lambda df: (df.isna().mean()) * 100)
    .round(2)
)

completeness

,pm25,pm10,co,no2,so2,o3
location,,,,,,
"Bollaram Industrial Area, Hyderabad - TSPCB",0.35,0.49,26.67,25.40,25.55,1.27
"Central University, Hyderabad - TSPCB",0.58,1.35,28.51,28.57,28.57,0.39
"ICRISAT Patancheru, Hyderabad - TSPCB",0.51,0.70,28.96,29.02,28.96,0.06
"IDA Pashamylaram, Hyderabad - TSPCB",1.64,1.51,29.73,33.63,29.79,0.55
"Zoo Park, Hyderabad - TSPCB",2.23,5.14,22.24,23.06,22.24,5.14


# Step 10: Computing Air Quality Index (AQI)


The current dataset contains pollutant concentrations collected from multiple monitoring stations across Hyderabad, including:

- PM2.5
- PM10
- CO
- NO₂
- SO₂
- O₃

While these pollutant concentrations are informative individually, the **Air Quality Index (AQI)** provides a single standardized indicator that summarizes overall air quality and its potential health impacts.

## Why Compute AQI?

The Central Pollution Control Board (CPCB) defines AQI using pollutant-specific breakpoint tables. For each pollutant, a **sub-index** is calculated based on its measured concentration. The overall AQI for a given day and station is then defined as the **maximum of all pollutant sub-indices**.

Mathematically,

> **AQI = Maximum(SubIndex_PM2.5, SubIndex_PM10, SubIndex_CO, SubIndex_NO₂, SubIndex_SO₂, SubIndex_O₃)**

The pollutant responsible for the maximum sub-index is referred to as the **dominant pollutant**.

## Is this Reverse Engineering?

No.

Although AQI is computed from pollutant concentrations, the objective of this project is **not** to train a model that reproduces the AQI formula for the same day.

Instead:

- Historical AQI values are computed using the official CPCB methodology.
- These historical values serve as **ground truth labels**.
- In later stages, the machine learning model will use historical observations to **forecast future AQI values** (e.g., next-day or multi-day forecasts).

Therefore, AQI computation is a necessary preprocessing step for creating the target variable used in time-series forecasting.

## Outputs Generated

This step creates the following new columns:

- `aqi` – Computed Air Quality Index
- `aqi_category` – AQI category (Good, Satisfactory, Moderate, Poor, Very Poor, Severe)
- `dominant_pollutant` – Pollutant contributing the highest sub-index

The resulting dataset will be used for exploratory analysis, feature engineering, forecasting, explainability, and anomaly detection.

In [71]:


# ============================================================
# CPCB AQI Breakpoints (Units: µg/m³ for PM, NO2, SO2, O3, CO)
# ============================================================

BREAKPOINTS = {
    "pm25": [
        (0, 30, 0, 50),
        (30, 60, 51, 100),
        (60, 90, 101, 200),
        (90, 120, 201, 300),
        (120, 250, 301, 400),
        (250, 500, 401, 500),
    ],
    "pm10": [
        (0, 50, 0, 50),
        (50, 100, 51, 100),
        (100, 250, 101, 200),
        (250, 350, 201, 300),
        (350, 430, 301, 400),
        (430, 600, 401, 500),
    ],
    "no2": [
        (0, 40, 0, 50),
        (40, 80, 51, 100),
        (80, 180, 101, 200),
        (180, 280, 201, 300),
        (280, 400, 301, 400),
        (400, 800, 401, 500),
    ],
    "so2": [
        (0, 40, 0, 50),
        (40, 80, 51, 100),
        (80, 380, 101, 200),
        (380, 800, 201, 300),
        (800, 1600, 301, 400),
        (1600, 2100, 401, 500),
    ],
    # OpenAQ CO values are already in µg/m³
    "co": [
        (0, 1000, 0, 50),
        (1000, 2000, 51, 100),
        (2000, 10000, 101, 200),
        (10000, 17000, 201, 300),
        (17000, 34000, 301, 400),
        (34000, 50000, 401, 500),
    ],
    "o3": [
        (0, 50, 0, 50),
        (50, 100, 51, 100),
        (100, 168, 101, 200),
        (168, 208, 201, 300),
        (208, 748, 301, 400),
        (748, 1000, 401, 500),
    ],
}

# ============================================================
# Function to compute AQI sub-index
# ============================================================

def compute_sub_index(value, breakpoints):
    """
    Compute AQI sub-index using CPCB linear interpolation.
    Values above the highest breakpoint are capped at AQI = 500.
    """

    if pd.isna(value):
        return np.nan

    # Cap values above the maximum supported concentration
    max_conc = breakpoints[-1][1]
    value = min(value, max_conc)

    for c_low, c_high, aqi_low, aqi_high in breakpoints:
        if c_low <= value <= c_high:
            return (
                ((aqi_high - aqi_low) / (c_high - c_low))
                * (value - c_low)
                + aqi_low
            )

    return np.nan


# ============================================================
# Create working copy
# ============================================================

aqi_df = station_filtered.copy()

# ============================================================
# Compute pollutant sub-indices
# ============================================================

for pollutant, bp in BREAKPOINTS.items():
    aqi_df[f"{pollutant}_si"] = aqi_df[pollutant].apply(
        lambda x: compute_sub_index(x, bp)
    )

# ============================================================
# Compute overall AQI
# ============================================================

subindex_cols = [
    "pm25_si",
    "pm10_si",
    "co_si",
    "no2_si",
    "so2_si",
    "o3_si",
]

aqi_df["aqi"] = aqi_df[subindex_cols].max(axis=1, skipna=True)

# ============================================================
# AQI Category
# ============================================================

def get_category(aqi):
    if pd.isna(aqi):
        return np.nan
    elif aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Satisfactory"
    elif aqi <= 200:
        return "Moderate"
    elif aqi <= 300:
        return "Poor"
    elif aqi <= 400:
        return "Very Poor"
    else:
        return "Severe"

aqi_df["aqi_category"] = aqi_df["aqi"].apply(get_category)

# ============================================================
# Dominant Pollutant
# ============================================================

aqi_df["dominant_pollutant"] = (
    aqi_df[subindex_cols]
    .idxmax(axis=1)
    .str.replace("_si", "", regex=False)
)

# ============================================================
# Save dataset
# ============================================================

aqi_df.to_csv(
    "data/processed/hyderabad_station_aqi.csv",
    index=False
)

# ============================================================
# Summary
# ============================================================

print("=" * 50)
print("AQI Dataset Created Successfully")
print("=" * 50)

print(f"Dataset Shape: {aqi_df.shape}")

print("\nAQI Statistics:")
print(aqi_df["aqi"].describe().round(2))

print("\nAQI Category Distribution:")
print(aqi_df["aqi_category"].value_counts())

print("\nDominant Pollutant Distribution:")
print(aqi_df["dominant_pollutant"].value_counts())

print("\nMissing AQI Values:")
print(aqi_df["aqi"].isna().sum())

print("\nSaved to:")
print("data/processed/hyderabad_station_aqi.csv")

AQI Dataset Created Successfully
Dataset Shape: (8077, 17)

AQI Statistics:
count    8077.00
mean       94.67
std        65.70
min         0.00
25%        54.76
50%        85.20
75%       119.48
max       500.00
Name: aqi, dtype: float64

AQI Category Distribution:
aqi_category
Satisfactory    3180
Moderate        2967
Good            1729
Severe           117
Very Poor         48
Poor              36
Name: count, dtype: int64

Dominant Pollutant Distribution:
dominant_pollutant
pm10    4769
pm25    1764
o3       518
no2      484
co       325
so2      217
Name: count, dtype: int64

Missing AQI Values:
0

Saved to:
data/processed/hyderabad_station_aqi.csv


# Understanding the Newly Created AQI Columns

After computing the Air Quality Index (AQI), several new columns were added to the dataset. These columns provide intermediate calculations and useful interpretations of the overall air quality.

### 1. Pollutant Sub-Indices (`*_si`)

For each pollutant, a corresponding AQI sub-index is calculated using the CPCB breakpoint tables.

The generated columns are:

- `pm25_si` – AQI sub-index for PM2.5
- `pm10_si` – AQI sub-index for PM10
- `co_si` – AQI sub-index for Carbon Monoxide (CO)
- `no2_si` – AQI sub-index for Nitrogen Dioxide (NO₂)
- `so2_si` – AQI sub-index for Sulfur Dioxide (SO₂)
- `o3_si` – AQI sub-index for Ozone (O₃)

These values represent the pollution severity contributed by each pollutant independently.

### 2. `aqi`

The final Air Quality Index is calculated as the **maximum** of all pollutant sub-indices.

```
AQI = max(pm25_si, pm10_si, co_si, no2_si, so2_si, o3_si)
```

This follows the official CPCB methodology, where the pollutant with the highest health risk determines the overall AQI.

### 3. `aqi_category`

The AQI value is mapped to a qualitative category:

| AQI Range | Category |
|-----------|----------|
| 0 – 50 | Good |
| 51 – 100 | Satisfactory |
| 101 – 200 | Moderate |
| 201 – 300 | Poor |
| 301 – 400 | Very Poor |
| 401 – 500 | Severe |

These categories make AQI easier to interpret for public communication.

### 4. `dominant_pollutant`

This column identifies the pollutant with the highest sub-index on a given day.

For example:

| PM2.5 SI | PM10 SI | CO SI | AQI | Dominant Pollutant |
|----------:|---------:|------:|----:|--------------------|
| 180 | 110 | 75 | 180 | PM2.5 |
| 140 | 120 | 430 | 430 | CO |

This helps explain **which pollutant is primarily responsible for poor air quality**.

### Data Quality Check: High Carbon Monoxide (CO) Values

During exploratory analysis, a small number of observations exhibited unusually high CO concentrations, with values exceeding 70,000 µg/m³.

Further inspection showed:

- Only 155 records exceeded 10,000 µg/m³.
- Only 114 records exceeded 34,000 µg/m³.
- These represent less than 2% of the dataset.
- The observations are concentrated in specific stations and dates rather than being randomly distributed.

Because these measurements may correspond to genuine high-pollution episodes, they were retained in the dataset and incorporated into AQI computation.

In [72]:
aqi_df.head()

,date,location,co,no2,o3,pm10,pm25,so2,pm25_si,pm10_si,no2_si,so2_si,co_si,o3_si,aqi,aqi_category,dominant_pollutant
0,2018-03-09,"Bollaram Industrial Area, Hyderabad - TSPCB",51500.0,40.0,38.0,118.0,106.0,10.0,253.80,112.880,50.000,12.50,500.000000,38.000,500.000000,Severe,co
1,2018-03-10,"Bollaram Industrial Area, Hyderabad - TSPCB",43200.0,44.8,33.8,121.0,108.0,10.0,260.40,114.860,56.880,12.50,457.925000,33.800,457.925000,Severe,co
2,2018-03-11,"Bollaram Industrial Area, Hyderabad - TSPCB",35200.0,45.2,52.4,135.0,146.0,12.2,320.80,124.100,57.370,15.25,408.425000,53.352,408.425000,Severe,co
3,2018-03-12,"Bollaram Industrial Area, Hyderabad - TSPCB",23000.0,25.3,44.4,97.1,81.9,8.0,173.27,97.158,31.625,10.00,335.941176,44.400,335.941176,Very Poor,co
4,2018-03-13,"Bollaram Industrial Area, Hyderabad - TSPCB",27700.0,34.0,26.3,92.0,66.3,7.0,121.79,92.160,42.500,8.75,363.311765,26.300,363.311765,Very Poor,co


# Final Outputs

### After executing all preprocessing and AQI computation steps, the following datasets are generated:

* **`locations.csv`** → Metadata for all discovered air quality monitoring stations in Hyderabad.
* **`sensors.csv`** → Metadata for pollutant sensors associated with each monitoring station.
* **`hyderabad_raw.csv`** → Raw historical pollutant observations downloaded from the OpenAQ API in long format.
* **`hyderabad_wide.csv`** → Daily city-level pivoted dataset with one column per pollutant.
* **`hyderabad_station_level.csv`** → Daily station-level dataset containing pollutant concentrations for each monitoring station.
* **`hyderabad_station_filtered.csv`** → Filtered station-level dataset containing only high-quality monitoring stations selected based on data completeness.
* **`hyderabad_station_aqi.csv`** → Final processed dataset containing pollutant concentrations, AQI sub-indices, computed AQI values, AQI categories, and dominant pollutants for each station and day.

### Key Columns Added in `hyderabad_station_aqi.csv`

* **`pm25_si`** → AQI sub-index computed from PM2.5 concentration.
* **`pm10_si`** → AQI sub-index computed from PM10 concentration.
* **`co_si`** → AQI sub-index computed from Carbon Monoxide (CO).
* **`no2_si`** → AQI sub-index computed from Nitrogen Dioxide (NO₂).
* **`so2_si`** → AQI sub-index computed from Sulfur Dioxide (SO₂).
* **`o3_si`** → AQI sub-index computed from Ozone (O₃).
* **`aqi`** → Final Air Quality Index obtained as the maximum of all pollutant sub-indices.
* **`aqi_category`** → Air quality category (Good, Satisfactory, Moderate, Poor, Very Poor, or Severe).
* **`dominant_pollutant`** → Pollutant contributing the highest AQI sub-index for that observation.

### These datasets provide a complete foundation for:

* Exploratory Data Analysis (EDA)
* Time-series analysis
* AQI trend visualization
* Pollution hotspot identification
* Seasonal and temporal pattern analysis
* Explainability and interpretation of AQI drivers
* Anomaly detection and unusual pollution event analysis
* Future AQI forecasting and machine learning model development
